In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import pandas as pd
from keras import backend as K
from keras.models import Model
from keras.regularizers import l2
from keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, concatenate, BatchNormalization, Flatten
)
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

In [ ]:
print(tf.__version__)

In [ ]:
import pandas as pd

# Load train and test datasets
X_train_downsampled = pd.read_csv('data/readmit/X_train_downsampled.csv')
y_train_downsampled = pd.read_csv('data/readmit/y_train_downsampled.csv')
y_test = pd.read_csv('data/readmit/y_test.csv')
X_test = pd.read_csv('data/readmit/X_test.csv')


In [ ]:
print(X_train_downsampled)

In [ ]:
column_names = X_train_downsampled.columns.tolist()
print(column_names)


In [ ]:
import pickle
# Load the LabelEncoder for ICD codes
with open('./Model/readmit_2016_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

In [ ]:
# Get the actual number of unique ICD codes
num_unique_icd_codes = len(encoder.classes_)
print("Number of unique ICD codes:", num_unique_icd_codes)

In [ ]:
from keras.saving import register_keras_serializable

### Pretrained Embeddings

In [ ]:
embeddings_50 = pd.read_csv('embeddings/icd-10-cm-2019-0050.csv')
embeddings_50

In [ ]:
embeddings_50 = embeddings_50.set_index('code')
embeddings_50

In [ ]:
#    assume columns e1…e50 (and maybe a “desc” you drop):
embeddings_50 = embeddings_50.drop(columns=['desc'], errors='ignore')

In [ ]:
codes = list(encoder.classes_) 
emb_df = pd.DataFrame(index=codes, columns=embeddings_50.columns, dtype=float)
emb_df

In [ ]:
# 3) Fill in exact matches
common = emb_df.index.intersection(embeddings_50.index)
emb_df.loc[common] = embeddings_50.loc[common]
emb_df

In [ ]:
missing = emb_df.isna().any(axis=1)
if missing.any():
    print("⚠️ Missing embeddings for these ICD codes:")
    print(emb_df.index[missing].tolist())

In [ ]:
# Set the null row to be zero vector
emb_df.loc[""] = 0.

# 4) For each “missing” code, find all more detailed codes and average
missing = emb_df.index[emb_df.isna().any(axis=1)]

for coarse in missing:
    # look for any base codes that start with the coarse code string:
    matches = embeddings_50.index[embeddings_50.index.str.startswith(coarse)]
    if len(matches):
        emb_df.loc[coarse] = embeddings_50.loc[matches].mean(axis=0)

missing = emb_df.isna().any(axis=1)
if missing.any():
    print("⚠️ Missing embeddings for these ICD codes:")
    print(emb_df.index[missing].tolist())

In [ ]:
# 4) Build a list of all base codes for fast lookup
base_codes = list(embeddings_50.index)

# 4) For each “missing” code, find all more detailed codes and average
missing = emb_df.index[emb_df.isna().any(axis=1)]

for fine in missing:
    # find all base codes that are a prefix of the fine code
    prefixes = [b for b in base_codes if fine.startswith(b)]
    if prefixes:
        # pick the longest matching prefix
        best = max(prefixes, key=len)
        emb_df.loc[fine] = embeddings_50.loc[best]
        
missing = emb_df.isna().any(axis=1)
if missing.any():
    print("⚠️ Missing embeddings for these ICD codes:")
    print(emb_df.index[missing].tolist())

In [ ]:
missing = emb_df.index[emb_df.isna().any(axis=1)]
for each in missing:
    emb_df.loc[each] = 0.
    
missing = emb_df.isna().any(axis=1)
if missing.any():
    print("⚠️ Missing embeddings for these ICD codes:")
    print(emb_df.index[missing].tolist())
else: 
    print("no missing embeddings")

In [ ]:
import numpy as np
embedding_matrix = emb_df.values.astype(np.float32)
assert embedding_matrix.shape == (num_unique_icd_codes, 50)
embedding_matrix

### Transformers

In [ ]:
import tensorflow as tf
from keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, concatenate, MultiHeadAttention, LayerNormalization, Add
from keras.models import Model
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
import keras.backend as K

# Custom Transformer Encoder Block
@tf.keras.utils.register_keras_serializable(package="Custom")
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        # Return the parameters required for serialization
        config = super(TransformerBlock, self).get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Recreate the layer from its config
        return cls(**config)
    

# class ResidualBlock(tf.keras.layers.Layer):
#     def __init__(self, units, dropout_rate=0.2):
#         super(ResidualBlock, self).__init__()
#         self.dense1 = Dense(units, activation='relu', kernel_regularizer=l2(0.001))
#         self.batch_norm1 = BatchNormalization()
#         self.dropout1 = Dropout(dropout_rate)

#         self.dense2 = Dense(units, kernel_regularizer=l2(0.001))
#         self.batch_norm2 = BatchNormalization()
#         self.dropout2 = Dropout(dropout_rate)

#         self.shortcut_dense = Dense(units) if units else None  # Use a dense layer if needed to match dimensions

#     def call(self, inputs, training=False):
#         x = self.dense1(inputs)
#         x = self.batch_norm1(x, training=training)
#         x = self.dropout1(x, training=training)

#         x = self.dense2(x)
#         x = self.batch_norm2(x, training=training)
#         x = self.dropout2(x, training=training)

#         # Adjust input dimensions if necessary
#         shortcut = self.shortcut_dense(inputs) if self.shortcut_dense else inputs
#         return tf.keras.layers.Add()([x, shortcut])   

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class DeepSet(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, output_dim, **kwargs):
        super(DeepSet, self).__init__(**kwargs)
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        # Element-wise transformation: phi network
        self.phi = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu')
        ])
        # Post-aggregation transformation: rho network
        self.rho = tf.keras.Sequential([
            Dense(self.output_dim, activation='relu')
        ])

    
    def call(self, x):
        # Apply phi to each ICD code embedding
        transformed = self.phi(x)  # Shape: (batch_size, num_codes, output_dim)
        # Aggregate using sum (or other aggregation functions)
        aggregated = tf.reduce_sum(transformed, axis=1)  # Shape: (batch_size, output_dim)
        # Apply rho to the aggregated representation
        output = self.rho(aggregated)  # Shape: (batch_size, output_dim)
        return output

    def get_config(self):
        # Return the parameters required for serialization
        config = super(DeepSet, self).get_config()
        config.update({
            "input_dim": self.input_dim,
            "hidden_dim": self.hidden_dim,
            "output_dim": self.output_dim
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Recreate the layer from its config
        return cls(**config)


In [ ]:
icd_columns = [f'I10_DX{i}' for i in range(1, 36)]
icd_inputs = Input(shape=(len(icd_columns),), name='icd_codes')
icd_embedding = Embedding(
    input_dim=num_unique_icd_codes,
    output_dim=50,
    weights=[embedding_matrix],
    trainable=True,           # or True if you still want to fine-tune
    name='icd_pretrained_embedding'
)
x = icd_embedding(icd_inputs)

print(x.shape)

In [ ]:
# Define the ICD code input layer with trainable embeddings
icd_columns = [f'I10_DX{i}' for i in range(1, 36)]
icd_inputs = Input(shape=(len(icd_columns),), name='icd_codes')
icd_embedding = Embedding(
    input_dim=num_unique_icd_codes,
    output_dim=32,
    name='icd_embedding',
    trainable=True
)(icd_inputs)
print(icd_embedding.shape)

x = icd_embedding

In [ ]:
# Add Transformer layer to ICD embeddings
# num_transformer_blocks = ４  # Set the number of transformer blocks to stack
# for i in range(num_transformer_blocks):
#     transformer_block = TransformerBlock(embed_dim=32, num_heads=3, ff_dim=128, rate=0.4)
#     x = transformer_block(x, training=True)

agg_block = DeepSet(input_dim = 50, hidden_dim = 256, output_dim = 128)
x = agg_block(x) 
    
# # Flatten the transformer output for input into the MLP
# x = Flatten()(icd_embedding)

# # Define MLP layers after transformer encoder
# # MLP layers for ICD embeddings
# x = Dense(256, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_1')(x)
# x = Dropout(0.4, name='mlp_dropout_1')(x)
# x = Dense(128, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_2')(x)
# x = Dropout(0.4, name='mlp_dropout_2')(x)
# x = Dense(64, activation='relu', kernel_regularizer=l2(0.001), name='mlp_dense_3')(x)

# Define demographic and one-hot encoded inputs
age_input = Input(shape=(1,), name='age')
female_input = Input(shape=(1,), name='female')
pay1_inputs = [Input(shape=(1,), name=f'PAY1_{col}') for col in X_train_downsampled.filter(regex='PAY1_').columns]
zipinc_qrtl_inputs = [Input(shape=(1,), name=f'ZIPINC_QRTL_{col}') for col in X_train_downsampled.filter(regex='ZIPINC_QRTL_').columns]

# Concatenate all inputs
concatenated = concatenate([x, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, name='concatenate')

# Add BatchNormalization and dense layers
hidden = BatchNormalization(name='batch_norm')(concatenated)
hidden = Dense(128, activation='relu', kernel_regularizer=l2(0.001), name='dense_1')(hidden)
hidden = Dropout(0.3, name='dropout_1')(hidden)
hidden = Dense(64, activation='relu', kernel_regularizer=l2(0.001), name='dense_2')(hidden)
hidden = Dropout(0.3, name='dropout_2')(hidden)
hidden = Dense(32, activation='relu', kernel_regularizer=l2(0.001), name='dense_3')(hidden)
hidden = Dropout(0.3, name='dropout_3')(hidden)

# hidden = BatchNormalization(name='batch_norm')(concatenated)
# hidden = ResidualBlock(units=128, dropout_rate=0.3)(hidden)
# hidden = ResidualBlock(units=64, dropout_rate=0.3)(hidden)
# hidden = ResidualBlock(units=32, dropout_rate=0.2)(hidden)


# Output layer for mortality prediction
output = Dense(1, activation='sigmoid', name='output')(hidden)

# Build the model
model = Model(inputs=[icd_inputs, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, outputs=output)

In [ ]:
# # Custom F2 score metric
# def f2_score(y_true, y_pred):
#     y_pred = tf.cast(y_pred > 0.5, tf.float32)
#     tp = K.sum(y_true * y_pred)
#     fp = K.sum((1 - y_true) * y_pred)
#     fn = K.sum(y_true * (1 - y_pred))
#     precision = tp / (tp + fp + K.epsilon())
#     recall = tp / (tp + fn + K.epsilon())
#     f2 = (5 * precision * recall) / (4 * precision + recall + K.epsilon())
#     return f2

@tf.keras.utils.register_keras_serializable(package="Custom")
def f2_score(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    # Calculate true positives, false positives, and false negatives
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    # Calculate precision and recall
    epsilon = tf.keras.backend.epsilon()  # Small constant to avoid division by zero
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)

    # Calculate F2 score
    f2 = (5 * precision * recall) / (4 * precision + recall + epsilon)
    return f2

In [ ]:
import keras

# Build the model
model = Model(inputs=[icd_inputs, age_input, female_input] + pay1_inputs + zipinc_qrtl_inputs, outputs=output)

# Compile the model with the weighted loss function
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.binary_crossentropy,  # Use the custom weighted loss
    metrics=[AUC(name='auc'), Precision(name='precision'), Recall(name='recall'), f2_score]
)

# Implement early stopping
early_stopping = EarlyStopping(
    monitor='val_f2_score', patience=3, mode='max', restore_best_weights=True
)

# Train the model
history = model.fit(
    [X_train_downsampled[icd_columns], X_train_downsampled['AGE'], X_train_downsampled['FEMALE']] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='PAY1_').columns] + [X_train_downsampled[col] for col in X_train_downsampled.filter(regex='ZIPINC_QRTL_').columns],
    y_train_downsampled,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stopping]
)

In [ ]:

# Step 4: Evaluate the model
test_loss, test_auc, test_precision, test_recall, test_f2 = model.evaluate(
    [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in X_test.filter(regex='PAY1_').columns] + [X_test[col] for col in X_test.filter(regex='ZIPINC_QRTL_').columns], 
    y_test)

print(f'Test AUC: {test_auc:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test F2 Score: {test_f2:.4f}')


In [ ]:
optimal_threshold = 0
best_f1 = 0
for threshold in np.linspace(0, 1, 100):
    preds = (y_pred_prob > threshold).astype(int)
    score = f1_score(y_test, preds)
    if score > best_f1:
        best_f1 = score
        optimal_threshold = threshold
print(optimal_threshold)
print(best_f1)

In [ ]:

# Calculate F1-score
y_pred_prob = model.predict(
    [X_test[icd_columns], X_test['AGE'], X_test['FEMALE']] + [X_test[col] for col in X_test.filter(regex='PAY1_').columns] + [X_test[col] for col in X_test.filter(regex='ZIPINC_QRTL_').columns]
)
y_pred = (y_pred_prob > 0.5).astype(int)

f1 = f1_score(y_test, y_pred)
print(f'Test F1 Score: {f1:.4f}')

In [ ]:
import pickle
import keras

# # Save the trained model
model.save('Model/readmit_pretrained_embeddings_50_continue_train.keras')

# keras.saving.save_model(model, 'Model/mort_2016.keras')